## Evaluation pipeline for the microlane experiment

In [ ]:
# First consider all the variables
# The input image gets resized to a particular level
# Then create a pipeline to feed data into the model
# AFter this process is completed, then process the data
# Then after the processing is done find a way to take output from the model
# Then, convert the output to relevant format, and store it for future use
# Apply relevant computations

In [ ]:
# Imports of the Core Packages
import yaml, random
from tqdm.notebook import tqdm

In [ ]:
# Import custom libraries located at different folder location + configs
from microlane.utils.load_image import load_image_from_sample, parse_prediction
from microlane.utils.create_settings import create_settings
from microlane.utils.experiment import ExperimentEvaluate
from microlane.utils.loaders import load_dataset, load_model

In [ ]:
# First Load the Configuation file
with open("../configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

### Experiment Settings

In [ ]:
INFERENCE_VIS_NUMBER = 5
SAMPLE_NUMBER = 500
MODEL = "motion_blur"
DATASET = "tusimple"
AUGMENTATION_TYPE = "normal"
EXPERIMENT_NAME = f"{MODEL}_{DATASET}_{AUGMENTATION_TYPE}"
AUGMENTATION_SETTINGS = {}
OUTPUT_LOCATION = "/home/suyog/desktop/projects/microlane/results/EXPERIMENT/tusimple/lanenet2/motion_blur"


### Pre Processing Part

In [ ]:
## Load the Corresponding Model and the dataset based on given settings

data  = load_dataset(DATASET, config, SAMPLE_NUMBER)

model = load_model(MODEL, config)


In [ ]:
experiment = ExperimentEvaluate(
    
    experiment_name=EXPERIMENT_NAME,
    
    output_dir=OUTPUT_LOCATION
    
)

### Models and Datasets Loaded, Now Processing Part

In [ ]:
# Print some basic information of our data

print(f"Total items: {len(data)}\n")

item = data[0]
print(f"Image Path   : {item.image_path}")
print(f"h_samples    : {item.h_samples}")
print(f"lanes        : {item.lanes}")

In [ ]:
blur_range = tuple(config['data']['augmentation']['blur'])
rotation_range = tuple(config['data']['augmentation']['blur'])
zoom_range = tuple(config['data']['augmentation']['zoom'])
lighting_range =tuple(config['data']['augmentation']['lighting'])

In [ ]:
numbers = [random.randint(0, SAMPLE_NUMBER - 1) for _ in range(INFERENCE_VIS_NUMBER)]
print(numbers)

In [ ]:
PRESETS = {
    "normal":       dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0, motion_blur=0.0),
    "lighting-d":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=-0.4,  motion_blur=0.0), 
    "lighting-b":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.4,  motion_blur=0.0),
    "motion-blur":  dict(blur=0.0, rotation=0.0, zoom=1.0, lighting=0.0,  motion_blur=1),
    "camera-shake": dict(blur=1, rotation=10, zoom=1.1, lighting=-0.1,  motion_blur=1),
}


AUGMENTATION_SETTINGS = PRESETS[AUGMENTATION_TYPE]

print(f"Augmentation Type:      {AUGMENTATION_TYPE}")
print(f"Blur:                   {item.blur:.2f}")
print(f"Rotation:               {item.rotation:.2f}")
print(f"Zoom:                   {item.zoom:.2f}")
print(f"Lighting:               {item.lighting:.2f}")
print(f"Motion blur:            {item.motion_blur:.2f}")


In [ ]:
settings = create_settings(
    inference_vis_number=INFERENCE_VIS_NUMBER,
    sample_number=SAMPLE_NUMBER,
    model=MODEL,
    dataset=DATASET,
    augmentation_type=AUGMENTATION_TYPE,
    augmentation_settings=AUGMENTATION_SETTINGS,
    output_path=experiment.folder_dir,
    experiment_name=EXPERIMENT_NAME,
)

### Looping through all the testing examples

In [ ]:
for index, testing_example in enumerate(tqdm(data)):

    testing_example.blur        = AUGMENTATION_SETTINGS["blur"]
    testing_example.rotation    = AUGMENTATION_SETTINGS["rotation"]
    testing_example.zoom        = AUGMENTATION_SETTINGS["zoom"]
    testing_example.lighting    = AUGMENTATION_SETTINGS["lighting"]
    testing_example.motion_blur = AUGMENTATION_SETTINGS["motion_blur"]

    loaded_image = load_image_from_sample(testing_example)
    
    response = model.predict(loaded_image)
    
    modeloutput = parse_prediction(response)
    
    experiment.store_prediction(modeloutput)
        
    if index in numbers:
        
        experiment.visualize_prediction(modeloutput)
    
    if (index % 100 == 0) and (index != 0):
        
        print(f"Routine Container Restart for Addressing Memory Leak [{int(index/100)}]")
        
        model.container_manager.restart_container()